# Retrieval Sanity Check

End-to-end test of `src/retrieval.py`. Give it a prompt and it returns the top-5 papers by abstract cosine similarity, each paired with the best-scoring chunks and figure captions from the same paper.

The retrieval is implemented from scratch (numpy primitives only — see `cosine_similarity` in `src/retrieval.py`).


## Load embeddings + index + encoder


In [1]:
import json
import os
import sys

import numpy as np

sys.path.insert(0, os.path.join(os.getcwd(), ".."))
import config
from src.encoder import load_model
from src.retrieval import retrieve

EMBEDDINGS_DIR = config.EMBEDDINGS_DIR

abstracts = np.load(os.path.join(EMBEDDINGS_DIR, "abstracts.npy"))
chunks    = np.load(os.path.join(EMBEDDINGS_DIR, "chunks.npy"))
captions  = np.load(os.path.join(EMBEDDINGS_DIR, "captions.npy"))

with open(os.path.join(EMBEDDINGS_DIR, "index.json")) as f:
    index = json.load(f)

encoder_model_name = index.get("encoder_model", config.ENCODER_MODEL_NAME)

print(f"Encoder model : {encoder_model_name}")
print(f"abstracts.npy : {abstracts.shape}  ({abstracts.shape[0]} papers)")
print(f"chunks.npy    : {chunks.shape}  ({chunks.shape[0]} chunks)")
print(f"captions.npy  : {captions.shape}  ({captions.shape[0]} captions)")

# Defensive check — retrieval relies on all three matrices being in the same space
assert abstracts.shape[1] == chunks.shape[1] == captions.shape[1], (
    f"Embedding dim mismatch: abstracts={abstracts.shape[1]} chunks={chunks.shape[1]} captions={captions.shape[1]}"
)

# Load the encoder that built these matrices. If this fails with a dimension
# mismatch later, re-run `python scripts/build_embeddings.py --rebuild`.
model = load_model(encoder_model_name)


/mnt/c/Users/xanca/Repositories/CSCI473_Project/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Encoder model : all-mpnet-base-v2
abstracts.npy : (273, 768)  (273 papers)
chunks.npy    : (7634, 768)  (7634 chunks)
captions.npy  : (2686, 768)  (2686 captions)


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Loading weights:  43%|████▎     | 85/199 [00:00<00:00, 849.70it/s]

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 1292.74it/s]


MPNetModel LOAD REPORT from: sentence-transformers/all-mpnet-base-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


## Run a query

`retrieve()` does two-stage search:
1. Cosine k-NN over the abstract matrix → top-k paper IDs.
2. For each retrieved paper, score its chunks and captions against the same query vector and keep the best `top_chunks_per_paper` / `top_captions_per_paper` of each.

Edit `QUERY` below and re-run to test other prompts.


In [2]:
QUERY = "reinforcement learning from human feedback for language model alignment"
K = 5

result = retrieve(
    QUERY,
    abstracts,
    chunks,
    captions,
    index,
    k=K,
    model=model,
    top_chunks_per_paper=3,
    top_captions_per_paper=2,
)

print(f"Query: {QUERY!r}\n")
print(f"Top {K} papers (by abstract cosine):")
for rank, (pid, score) in enumerate(zip(result["paper_ids"], result["scores"]), 1):
    title = next(
        (c["paper_title"] for c in index["chunks"] if c["paper_id"] == pid),
        pid,
    )
    print(f"  {rank}. [{score:.4f}]  {title}  ({pid})")


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches: 100%|██████████| 1/1 [00:00<00:00,  3.06it/s]

Batches: 100%|██████████| 1/1 [00:00<00:00,  3.06it/s]

Query: 'reinforcement learning from human feedback for language model alignment'

Top 5 papers (by abstract cosine):
  1. [0.6177]  Merge and Conquer: Instructing Multilingual Models by Adding Target Language Weights  (2603.28263)
  2. [0.5274]  Visualization of Machine Learning Models through Their Spatial and Temporal Listeners  (2603.27527)
  3. [0.5176]  Understanding Teacher Revisions of Large Language Model-Generated Feedback  (2603.27806)
  4. [0.4957]  ATLAS-RTC: Closing the Loop on LLM Agent Output with Token-Level Runtime Control  (2603.27905)
  5. [0.4924]  Rethinking Easy-to-Hard: Limits of Curriculum Learning in Post-Training for Deductive Reasoning  (2603.27226)


In [3]:
from collections import defaultdict

print("Top chunks grouped by paper:\n")
by_paper = defaultdict(list)
for p in result["passages"]:
    by_paper[p["paper_id"]].append(p)

for pid in result["paper_ids"]:
    group = by_paper.get(pid, [])
    if not group:
        continue
    title = group[0]["paper_title"] or pid
    print(f"— {title} ({pid})")
    for p in group:
        snippet = p["text"][:120].strip().replace("\n", " ")
        print(f"    [{p['score']:.4f}] {p['heading']}")
        print(f"             {snippet}...")
    print()


Top chunks grouped by paper:

— Merge and Conquer: Instructing Multilingual Models by Adding Target Language Weights (2603.28263)
    [0.5860] Acknowledgments
             This work has been funded by the Spanish Ministry of Science, Innovation, and Universities (Project HumanAIze, grant num...
    [0.5835] 4.  Experimental Setup
             To thoroughly evaluate our hypotheses, we conducted experiments across multiple languages (Basque, Galician, Catalan, an...
    [0.5687] 7.  Limitations
             Despite the promising results, this work has several limitations. Our study is restricted to a limited number of model f...

— Visualization of Machine Learning Models through Their Spatial and Temporal Listeners (2603.27527)
    [0.3732] RETRIEVAL-AUGMENTED METHOD FOR HUMAN-LLM LITERATURE EXTRACTION
             The details of our literature extraction workflow are explained in this section. The core challenge is to map a large, w...
    [0.3647] Stage 3: Framework-aligned Four-field

In [4]:
if result["captions"]:
    print("Top captions:\n")
    for c in result["captions"]:
        caption_text = c["caption"][:200].strip().replace("\n", " ") if c["caption"] else "(no caption text — legacy index format; rebuild embeddings to populate)"
        print(f"  [{c['score']:.4f}] {c['title']} ({c['paper_id']})")
        print(f"         {caption_text}")
        print()
else:
    print("No captions returned for the top-K papers.")


Top captions:

  [0.5157] Merge and Conquer: Instructing Multilingual Models by Adding Target Language Weights (2603.28263)
         Figure 1: Proportion sweep ablation for Llama 3.1 8B merges across Iberian languages (EU, GL, CA, ES), comparing Linear (top row) and Task Arithmetic (bottom row). Each subplot shows the effect of var

  [0.4802] Understanding Teacher Revisions of Large Language Model-Generated Feedback (2603.27806)
         Figure 1: LLM-supported feedback generation. Screenshot of the teacher’s tag, evaluation, and feedback module. The red area shows the feedback generated after clicking the Generate Feedback button. Ad

  [0.4032] Understanding Teacher Revisions of Large Language Model-Generated Feedback (2603.27806)
         Figure 2: Cumulative distribution of the percentage of messages changed by teachers. The x-axis shows the percentage of teachers (sorted by change rate), and the y-axis shows the percentage of message

  [0.3462] Rethinking Easy-to-Hard: Limits of

## Confidence signal

When the top-1 abstract score sits close to the top-5 floor, retrieval is ambiguous and the reranker downstream has real work to do. When the gap is wide, the abstract-level match is confident.


In [5]:
scores = result["scores"]
if len(scores) >= 2:
    gap = scores[0] - scores[-1]
    print(f"Abstract score spread (top-1 minus top-{len(scores)}): {gap:.4f}")
    print(f"Top-1 score: {scores[0]:.4f}    Top-{len(scores)} score: {scores[-1]:.4f}")
    print("Wide gap → confident top match. Tight cluster → candidates for reranking.")


Abstract score spread (top-1 minus top-5): 0.1253
Top-1 score: 0.6177    Top-5 score: 0.4924
Wide gap → confident top match. Tight cluster → candidates for reranking.


## Try another query

Swap prompts without re-running setup — cells above have already loaded the model and matrices.


In [6]:
QUERY_2 = "diffusion models for image generation"
K = 5

result2 = retrieve(QUERY_2, abstracts, chunks, captions, index, k=K, model=model)

print(f"Query: {QUERY_2!r}\n")
for rank, (pid, score) in enumerate(zip(result2["paper_ids"], result2["scores"]), 1):
    title = next(
        (c["paper_title"] for c in index["chunks"] if c["paper_id"] == pid),
        pid,
    )
    print(f"  {rank}. [{score:.4f}]  {title}  ({pid})")

print()
for p in result2["passages"][:5]:
    snippet = p["text"][:120].strip().replace("\n", " ")
    print(f"  [{p['score']:.4f}] {p['paper_title']} / {p['heading']}")
    print(f"           {snippet}...")


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches: 100%|██████████| 1/1 [00:00<00:00, 49.88it/s]

Query: 'diffusion models for image generation'

  1. [0.5946]  ImagenWorld: Stress-Testing Image Generation Models with Explainable Human Evaluation on Open-ended Real-World Tasks  (2603.27862)
  2. [0.5589]  $R_\text{dm}$: Re-conceptualizing Distribution Matching as a Reward for Diffusion Distillation  (2603.28460)
  3. [0.5540]  On-the-fly Repulsion in the Contextual Space for Rich Diversity in Diffusion Transformers  (2603.28762)
  4. [0.5410]  Stepwise Credit Assignment for GRPO on Flow-Matching Models  (2603.28718)
  5. [0.5318]  Unrestrained Simplex Denoising for Discrete Data. A Non-Markovian Approach Applied to Graph Generation  (2603.28572)

  [0.6410] $R_\text{dm}$: Re-conceptualizing Distribution Matching as a Reward for Diffusion Distillation / 6Conclusion
           In this work, we present a novel framework for improving diffusion distillation by re-conceptualizing distribution match...
  [0.6353] ImagenWorld: Stress-Testing Image Generation Models with Explainable Human 